# REGPLEX_Local — Reproducible Scientific Tutorial

This notebook reproduces the implemented REGPLEX v13 workflow for **Low Perplexity Region (LPR)** detection from FASTA input.


## 1. Introduction
REGPLEX identifies local sequence-complexity depressions in DNA using a deterministic, training-free pipeline.


In [ ]:
from regplex_core import *
from motif_engine import compile_motifs, annotate_regions
from visualization import (
    plot_perplexity_landscape,
    plot_smoothed_perplexity,
    plot_pds_landscape,
    plot_region_ranking,
    plot_three_window,
)
import pandas as pd


## 2. Theory
Dinucleotide perplexity measures local uncertainty in adjacent-base transitions. Sustained drops in perplexity relative to nearby context define candidate low-perplexity regions.


In [ ]:
workflow = [
    "DNA",
    "Dinucleotide Perplexity (17 nt)",
    "Savitzky–Golay Smoothing",
    "Perplexity Depression Score (PDS)",
    "Bounded Kadane Optimization",
    "Region Expansion",
    "Region Merging",
    "Low Perplexity Region Ranking",
    "Optional Motif Annotation",
    "Downloads",
]
workflow


## 3. Mathematical formulation
For upstream mean $\bar U$, region mean $\bar R$, and downstream mean $\bar D$:

$$
\mathrm{PDS}=\frac{\bar U + \bar D}{2}-\bar R
$$

Ranking score:

$$
\mathrm{RegionScore}=\mathrm{PDSMean}\times\mathrm{Persistence}\times\log(\mathrm{Length})\times\frac{1}{\mathrm{Variance}+\varepsilon}
$$


In [ ]:
print('PDS and RegionScore are computed directly in regplex_core.py')


## 4. Loading FASTA
Load input sequences and inspect record counts.


In [ ]:
with open('examples/ecoli.fasta', encoding='utf-8') as fh:
    fasta_text = fh.read()

records = parse_fasta(fasta_text)
len(records), records[0][0], len(records[0][1])


## 5. Dinucleotide perplexity
Compute the primary signal (window = 17 nt).


In [ ]:
sequence_id, seq = records[0]
di = compute_di_perplexity(seq, window=17)
len(di), float(pd.Series(di).dropna().mean())


## 6. Savitzky–Golay smoothing
Apply one smoothing pass to the perplexity profile.


In [ ]:
smoothed = smooth_perplexity(di, window_length=21, poly_order=3)
plot_smoothed_perplexity(di, smoothed, [])


## 7. PDS calculation
Compute local bilateral contrast against genomic flanks.


In [ ]:
pds = compute_pds(smoothed, flank_size=100, spacer_size=50, min_candidate=50, max_candidate=1000)
plot_pds_landscape(pds, [])


## 8. Region detection
Identify positive-PDS core regions using bounded Kadane optimization.


In [ ]:
cores = _find_all_pds_regions(pds, min_region_length=100, max_region_length=1000)
cores[:5], len(cores)


## 9. Expansion
Expand each core while local support remains above expansion rules.


In [ ]:
expanded = [_expand_region_pds(pds, s, e) for s, e in cores]
expanded[:5]


## 10. Merging
Merge nearby expanded regions separated by short gaps.


In [ ]:
merged = _merge_regions(expanded, gap=100)
merged[:5], len(merged)


## 11. Ranking
Compute region metrics and rank by RegionScore.


In [ ]:
result = analyze_sequence(sequence_id, seq)
df = pd.DataFrame(result.regions)
df[['Region_ID', 'Start', 'End', 'Length', 'PDSMean', 'RegionScore', 'Rank']].head()


## 12. Motif annotation
Annotate detected regions with optional motif patterns.


In [ ]:
motifs = compile_motifs('TATAWAWR\nCGCG')
annotated = annotate_regions(result.regions, motifs)
pd.DataFrame(annotated)[['Region_ID', 'MotifCount', 'Motifs']].head()


## 13. Export
Export tabular and interval outputs.


In [ ]:
export_df = pd.DataFrame(result.regions)
csv_bytes = export_table(export_df, 'csv')
bed_bytes = export_bed(export_df)
len(csv_bytes), len(bed_bytes)


## 14. Interpretation
Detected regions are algorithmic candidates defined by local complexity depression and ranking statistics. Biological significance requires independent experimental or comparative validation.


In [ ]:
plot_region_ranking(result.regions)
